# Class 20: The Registrar's Report
*Elements of Data Science (Honors)*

<br>**<center>Learning Goals**

|Area|Concept|
|---|---|
|Survivorship bias|A statistic computed only from cases that "made it" can misrepresent the whole population|
|Censoring|Cases still in progress (or lost) can't be scored as success or failure yet|
|Kaplan-Meier curve|A step-function estimate that correctly accounts for censored cases|

Today's question, in two parts: can you trust a statistic built only from the cases that succeeded? And if not, what's the honest way to compute it?

First, set up the imports by running the cell below.

In [ ]:
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')


## Part 1: Before We Look at Anything Else

Here is the registrar's report for one entering cohort: every student on record who has graduated, and how many years it took them.

In [ ]:
registrar = Table.read_table("./data/registrar_report.csv")
registrar

> **Handout Q1.1.** Before you run the next cell: does this number describe the typical experience of an entering student? Write your answer on the handout *first*.

In [ ]:
naive_mean = ...
naive_median = ...
print(f"Naive graduates-only mean: {naive_mean:.2f} years")
print(f"Naive graduates-only median: {naive_median:.2f} years")

## Part 2: What the Report Left Out

The registrar's report only tracks students who finished. Here is the **entire** entering cohort those graduates came from -- everyone who started, including dropouts and students still enrolled.

In [ ]:
full_cohort = Table.read_table("./data/full_cohort.csv")
full_cohort.show(5)

In [ ]:
full_cohort.group('status')

> **Handout Q2.1.** Compute the honest completion rate: what fraction of the *entire* entering cohort actually graduated? Record it on the handout.

In [ ]:
grads = full_cohort.where('status', 'graduate')
pct_graduated = ...
print(f"Honest completion rate: {pct_graduated:.1f}%")

In [ ]:
full_cohort.group('status').barh('status')

> **Handout Q2.2 Discussion.** What question does the naive average answer, and what question does it fail to answer at all?

## Part 3: The Proper Way to Count

Averaging only the finishers has a second, sneakier problem: even students who are *on track* to graduate don't count yet if we ask today. Statisticians call this **censoring** -- we know how long we've tracked a student, but not whether or when they'll graduate. There's a standard tool built for exactly this situation -- a **Kaplan-Meier curve**. Your instructor will walk through this one, built from scratch using only Table methods and numpy.

In [ ]:
def kaplan_meier(years, graduated):
    """Kaplan-Meier estimate of P(not yet graduated) at each observed graduation time.
    years: years tracked (event time if graduated, censoring time if not)
    graduated: 1 if the student graduated, 0 if dropped out or still enrolled (censored)
    Returns a Table with one row per graduation event.
    """
    years = np.array(years)
    graduated = np.array(graduated)

    # The distinct years on which at least one student graduated
    event_times = np.unique(years[graduated == 1])

    # Start the curve at year 0, where 100% of the cohort has not yet graduated
    time_points = make_array(0.0)
    survival = make_array(1.0)
    S = 1.0

    for i in np.arange(len(event_times)):
        t = event_times[i]
        at_risk = np.sum(years >= t)
        grads_at_t = np.sum((years == t) & (graduated == 1))
        S = S * (1 - grads_at_t / at_risk)
        time_points = np.append(time_points, t)
        survival = np.append(survival, S)

    return Table().with_columns('years', time_points, 'fraction_not_graduated', survival)

km_table = kaplan_meier(full_cohort.column('years_tracked'), full_cohort.column('graduated'))
km_table

In [ ]:
for i in np.arange(km_table.num_rows):
    year = km_table.column('years')[i]
    frac = km_table.column('fraction_not_graduated')[i]
    print(f"By year {year}: {100*frac:.1f}% of the entering cohort has not yet graduated")

In [ ]:
# Where do the censored students (dropouts and still-enrolled) sit on the curve?
# Each one leaves the "at risk" group without ever graduating -- that's censoring.
censored = full_cohort.where('graduated', 0)
censored_years = censored.column('years_tracked')

# For each censored student, find the survival level at the time they left --
# that's the most recent graduation-event value at or before their own year.
censor_levels = make_array()
for i in np.arange(len(censored_years)):
    c = censored_years[i]
    level = 1.0
    for j in np.arange(km_table.num_rows):
        if km_table.column('years')[j] <= c:
            level = km_table.column('fraction_not_graduated')[j]
    censor_levels = np.append(censor_levels, level)

num_dropout = censored.where('status', 'drop_out').num_rows
num_still_enrolled = censored.where('status', 'still_enrolled').num_rows
print(f"{len(censored_years)} students are censored: {num_dropout} dropped out, {num_still_enrolled} still enrolled")

Every dropout and every still-enrolled student is **censored** -- we know they haven't graduated *yet*, but we don't get to call it a failure either. The tick marks below show exactly where each censored student leaves the \"at risk\" group.

In [ ]:
plt.step(km_table.column('years'), km_table.column('fraction_not_graduated'),
         where='post', linewidth=2.5, color='#9E1B34', label='graduated (Kaplan-Meier)')
plt.plot(censored_years, censor_levels, marker='|', linestyle='None', markersize=14,
         color='steelblue', label='censored (dropped out or still enrolled)')
plt.axhline(0.5, linestyle=':', color='lightgray')
plt.axvline(4.31, linestyle='--', color='gray', label='naive graduates-only mean (4.31 yr)')
plt.xlabel('Years since enrollment')
plt.ylabel('Fraction not yet graduated')
plt.title('Kaplan-Meier: accounting for dropouts and students still enrolled')
plt.legend()
plt.show()

> **Handout Q3.1.** At year 6, what percentage of the *original* entering cohort still has not graduated? Is this knowable from the registrar's graduates-only report?

> **Handout Q3.2.** The curve puts the 50%-graduated point at 4.5 years, later than the naive average of 4.31 years. Which direction did the naive number get it wrong, and why?

## Part 4: Back to the Bombers (paper discussion, no code)

> **Handout Q4.1.** Fill in the Registrar's Report column of the analogy table, matching each row to today's activity.

> **Handout Q4.2.** Name one more place -- outside planes and college -- where only looking at the "survivors" would give you a misleading picture.